# Pipeline


Dataset : SNLI

Modèle : DistilBERT

Tâche : NLI


Objectif : prendre en main la méthode

# **Instalattion**

In [ ]:
# Décommenter seulement si on change de session Colab

#!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
#!pip install -q transformers accelerate peft datasets evaluate
#import os; os.kill(os.getpid(), 9)  # redémarrer après installation

In [ ]:
# Vérifier la compatibilité AVANT d'importer transformers
import torch
import torchvision
print(f"torch       : {torch.__version__}")
print(f"torchvision : {torchvision.__version__}")
print(f"CUDA dispo  : {torch.cuda.is_available()}")

In [ ]:
import json, os
import numpy as np
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt

from datasets import load_dataset         # Charger SNLI
from transformers import (
    AutoTokenizer,                        # Charge automatiquement le tokenizer adapté au modèle choisi.
    AutoModelForSequenceClassification,   # Charge automatiquement un modèle pré-entraîné pour la classification de texte.
    DataCollatorWithPadding,              # Gère le padding dynamique au moment de créer les batchs.
    TrainingArguments,                    # Permet de configurer tous les paramètres d'entraînement.
    Trainer,                              # Classe haut niveau qui gère tout le pipeline d'entraînement.
    EarlyStoppingCallback

    )
import evaluate                          # Métriques

# librairie PEFT (Parameter-Efficient Fine-Tuning.)
# Pour la méthode de fine-tuning LoRA
from peft import (
    LoraConfig,                          # Permet de configurer LoRA (Low-Rank Adaptation), une technique de fine‑tuning très légère.
    get_peft_model,                      # Fonction qui injecte la configuration LoRA dans un modèle HuggingFace existant.
    TaskType                             # Enumération qui indique le type de tâche (classification, génération, token classification…).
    )

import transformers, peft, accelerate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Vérification
print(f"transformers : {transformers.__version__}")
print(f"peft         : {peft.__version__}")
print(f"accelerate   : {accelerate.__version__}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

save_path= "/content/drive/MyDrive/PROJET_DJMA/models/distilbert_lora_snli_nli"


#  **Chargement et exploration des données**

In [ ]:
# Chargement du dataset SNLI
dataset = load_dataset("snli")
print(dataset)

# un exemple du dataset
print("exemple ", dataset["train"][0])

# Taille des splits
print("Train: ", len(dataset["train"]))
print("Validation: ", len(dataset["validation"]))
print("Test: ", len(dataset["test"]))

In [ ]:
# Voir des exemples invalides, les 5 premiers du train.
invalid_examples = dataset["train"].filter(lambda x: x["label"] == -1)
invalid_examples[:5]

In [ ]:
# Filtrer les labels invalides (-1)
dataset = dataset.filter(lambda x: x["label"] != -1)

print("Après filtrage:")
print("Train:", len(dataset["train"]))
print("Validation:", len(dataset["validation"]))
print("Test:", len(dataset["test"]))

In [ ]:
# Ajouter les labels textuels
label_text= {
    0: "entailment",
    1: "neutral",
    2: "contradiction"
}

def add_label_text(example):
  example["label_text"] = label_text[example["label"]]
  return example

dataset = dataset.map(add_label_text)

In [ ]:
for i in range(3):
    print(dataset["train"][i])
    print("-----")


In [ ]:
# Les distributions des classes pour tous les splits

splits = ["train", "validation", "test"]

for split in splits:
    labels = [x["label"] for x in dataset[split]]
    counts = Counter(labels)

    # Tracer le graphique
    plt.figure(figsize=(6, 4))
    bars = plt.bar([label_text[i] for i in counts.keys()], counts.values(), color=["#4C72B0", "#55A868", "#C44E52"] )


    # Ajouter les valeurs sur les barres
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, height, str(height),
                 ha="center", va="bottom")

    plt.title(f"Distribution des classes — {split}")
    plt.xlabel("Labels textuels")
    plt.ylabel("Nombre d'exemples")
    plt.show()


In [ ]:
# Analyser la longueur des séquences

# Calcul des longueurs en nombre de mots
premise_lengths = [len(x["premise"].split()) for x in dataset["train"]]
hypothesis_lengths = [len(x["hypothesis"].split()) for x in dataset["train"]]
total_lengths = [
    len((x["premise"] + " " + x["hypothesis"]).split())
    for x in dataset["train"]
]

# Construction d'un DataFrame
df_lengths = pd.DataFrame({
    "premise_length": premise_lengths,
    "hypothesis_length": hypothesis_lengths,
    "total_length": total_lengths
})

# Affichage des statistiques descriptives
df_lengths.describe()


Taille des prémisses (premise_length)
- Moyenne : 13 mots
- Min : 2 mots
- Max : 78 mots

Taille des hypothèses (hypothesis_length)
- Moyenne : 8 mots
- Min : 1 mots
- Max : 56 mots

Longueur totale (total_length)
- Moyenne : 20 mots
- Min : 4 mots
- Max : 112 mots

---

**Pourquoi la longueur de séquence est importante :**

**Les transformers ont une longueur maximale d’entrée. Si le texte est trop long : il est tronqué, perte d'information et si elle est trop courte : contexte insuffisant**

D’où l’importance de hyperparametre **max_length**.

# 2) **Charger le modèle et Tokenisation**

Le tokenizer est choisi de manière cohérente avec l’architecture du modèle transformer utilisé, indépendamment du jeu de données. En effet, chaque modèle possède son propre vocabulaire et ses tokens spécifiques, rendant nécessaire l’utilisation du tokenizer correspondant pour garantir une représentation correcte des séquences textuelles

In [ ]:
# Charger le modèle
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

- tokenizer_config.json : Contient la configuration du tokenizer. Par ex. (quels tokens spéciaux utiliser ([CLS], [SEP], [PAD])) ; (type de tokenizer (WordPiece, BPE, etc.))

- config.json : La configuration du modèle BERT lui-même.(nombre de couches, dropout, etc.)


- vocab.txt : Le vocabulaire WordPiece de BERT.

- tokenizer.json : Version optimisée et complète du tokenizer (Il permet un chargement plus rapide et plus efficace).


La tokenisation découpe une phrase en unités appelées tokens.

* Exemple: tokenizer.tokenize("unbelievable story") retourne -->
* ['un', '##bel', '##iev', '##able', 'story']


Le modèle ne comprend pas les mots, seulement des nombres. Le tokenizer convertit donc chaque token en un ID du vocabulaire WordPiece.

* Exemple : tokenizer("Hello world")
retourne -->

* {
  'input_ids': [101, 7592, 2088, 102],
  'attention_mask': [1, 1, 1, 1]
}

* 101 = token spécial [CLS] et 102 = token spécial [SEP]

* Le tokenizer génère un masque : 1 = token réel et 0 = padding. Cela permet au modèle d’ignorer les zones remplies artificiellement.

---

En NLI, on donne deux textes au modèle :
prémisse et hypothèse.

Ils sont fusionnés comme ceci :

[CLS] prémisse [SEP] hypothèse [SEP]

La tokenisation :
- encode les deux textes ensemble
- permet au modèle de comparer mot à mot les deux phrases
- rend possible l’inférence logique
---
Ce que produit la tokenisation
Pour chaque paire de textes, on obtient :

| Élément                         | Rôle                             |
| ------------------------------- | -------------------------------- |
| `input_ids`                     | Tokens numériques                |
| `attention_mask`                | Indique tokens utiles vs padding(remplissage artificiel) |
| `token_type_ids` (selon modèle) | Différencie texte A / B          |


In [ ]:
print(tokenizer.model_input_names)

In [ ]:
# Fonction de tokenisation

def tokenize(example):
    return tokenizer(
        example["premise"],
        example["hypothesis"],
        truncation=True,          # (nécessaire) coupe la phrase si elle depasse max_length sinon erreur.
        max_length=128,
        return_token_type_ids=True
        )

tokenized = dataset.map(
    tokenize,
    batched=True   # tokenise plusieurs exemples à la fois
  )

data_collator = DataCollatorWithPadding(tokenizer=tokenizer) # ajoute le padding dynamique au moment de créer un batch, Pas de padding inutile

In [ ]:
# Aperçu du dataset token
tokenized["train"][0]

In [ ]:
def inspect(idx):
    # On récupère un exemple tokenisé dans le split train
    example = tokenized["train"][idx]

    # IDs numériques des tokens
    ids = example["input_ids"]

    # Tokens WordPiece lisibles
    tokens = tokenizer.convert_ids_to_tokens(ids)
    # Mask : 1 = token réel, 0 = padding
    masks = example["attention_mask"]
    # segement A/B
    type_ids   = example["token_type_ids"]

    # Affichage du texte original
    print("\n------\n")
    print("Premise :", example["premise"])
    print("Hypothesis :", example["hypothesis"])
    print("\n--- Tokenisation détaillée ---\n")


    # Légende
    print(f"{'Token':<15} {'ID':>6} {'Mask':>6} {'Type':>6}   Segment")
    print("-" * 55)

    # Affichage token / id / mask
    for t, i, m, ty in zip(tokens, ids, masks, type_ids):
        segment = "Premise   " if ty == 0 else "Hypothesis"
        print(f"{t:<15} {i:>6} {m:>6} {ty:>6}   {segment}")

In [ ]:
inspect(1)
inspect(3)

## Préparer les données pour le modèle

Supprimer les colonnes textuelles, le modèle n’utilise pas le texte brut.

* Le Trainer attend : input_ids, attention_mask, token_type_ids et label.



In [ ]:
tokenized_dataset = tokenized.remove_columns(
    ["premise", "hypothesis", "label_text"]
)
tokenized_dataset

In [ ]:
tokenized_dataset["train"][3]

## Architecture DistilBertForSequenceClassification


In [ ]:
# Charger le modèle et LoRA
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
    )


Le modèle ajoute automatiquement : une tête de classification sur BERT et une loss CrossEntropy.

La tête de classification est initialisée aléatoirement et apprise durant le fine-tuning.


In [ ]:
print(model)

## LoRA (Low-Rank Adaptation)
C'est une technique dans le domaine de l'apprentissage automatique (ML) conçue pour affiner efficacement les modèles pré-entraînés massifs.


In [ ]:
# Configuration LoRA
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,        # Classification de séquences
    r=16,                              # Rang des matrices LoRA
    lora_alpha=32,                     # alpha est le facteur de mise à l'échelle pour les poids appris. La matrice de poids est mise à l'échelle par alpha/r.
    lora_dropout=0.1,                  # Dropout pour régularisation
    target_modules=["q_lin", "v_lin", "k_lin"], # Couches attention de DistilBERT
    bias="none",
    inference_mode=False
)

# Appliquer LoRA au modèle
model_lora = get_peft_model(model, lora_config)
print(model_lora.print_trainable_parameters())     # Vérifie combien de paramètres sont entraînés




 L’application de LoRA sur DistilBERT réduit le nombre de paramètres entraînables à environ 1 % du total, ce qui permet un fine‑tuning efficace et léger tout en conservant les performances du modèle.

In [ ]:
print(model_lora)

# 3) **Entraînement et évaluation**

In [ ]:
# Définir les métriques
accuracy = evaluate.load("accuracy")  # % de bonnes prédictions,performance globale
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1.compute(predictions=preds, references=labels, average="macro")["f1"],
    }


In [ ]:
# Calculer le warmup optimal
train_size = len(tokenized_dataset["train"])
batch_size = 32
num_epochs = 15

steps_per_epoch = train_size // batch_size
total_steps = steps_per_epoch * num_epochs
warmup_steps = int(0.1 * total_steps)  # 10% du training

print(f"  Configuration d'entraînement:")
print(f"  Exemples d'entraînement: {train_size:,}")
print(f"  Steps par epoch: {steps_per_epoch:,}")
print(f"  Total steps: {total_steps:,}")
print(f"  Warmup steps (10%): {warmup_steps:,}")

# Paramètres d’entraînement
training_args = TrainingArguments(
    seed=42,
    output_dir= save_path,              # Dossier où seront enregistrés les checkpoints et résultats

    # Stratégies d'évaluation et de sauvegarde
    eval_strategy="epoch",        # Évalue le modèle à la fin de chaque epoch (anciennement eval_strategy)
    save_strategy="epoch",              # Sauvegarde un checkpoint à la fin de chaque epoch
    save_total_limit=2,                 # garde seulement le best checkpoint
    load_best_model_at_end=True,        # Recharge automatiquement le meilleur modèle à la fin
    metric_for_best_model="eval_f1_macro", # Métrique utilisée pour déterminer le meilleur modèle
    greater_is_better=True,             # Plus la métrique est élevée, meilleur est le modèle

    # Hyperparamètres d'entraînement
    learning_rate=2e-4,                         # Taux d’apprentissage
    per_device_train_batch_size=batch_size,     # Taille des batchs pour l’entraînement (par GPU)
    per_device_eval_batch_size=64,              # Taille des batchs pour l’évaluation
    num_train_epochs=num_epochs,                # Nombre total d’epochs
    weight_decay=0.01,                          # Régularisation L2 pour éviter l’overfitting
    warmup_steps=warmup_steps,                  # warmup améliore la stabilité et la convergence
    optim="adamw_torch",

    # Logging
    logging_strategy="steps",           # Log toutes les X étapes
    logging_steps=100,                  # Fréquence des logs (toutes les 100 étapes)

    # Optimisations
    fp16=True,                          # Active le half-precision (16 bits) si GPU compatible
    report_to="none",                   # Désactive les logs vers WandB / TensorBoard
)

### Trainer :
gère la boucle d’entraînement, le backprop, l'évaluation périodique et les métriques


In [ ]:
trainer_lora = Trainer(
    model=model_lora,
    args=training_args,
    train_dataset= tokenized_dataset["train"],
    eval_dataset= tokenized_dataset["validation"],
    processing_class=tokenizer,
    #tokenizer=tokenizer,                           # depend des versions
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

# Lancer l'entrainement, évaluation pendant l’apprentissage
trainer_lora.train()


In [ ]:
batch = next(iter(trainer_lora.get_train_dataloader()))
print(batch.keys())

## **'token_type_ids' ne sont PAS utilisés !**
Pourquoi ? Parce que DistilBERT n'utilise PAS les token_type_ids en interne, même si le tokenizer les génère. C'est une simplification architecturale de DistilBERT par rapport à BERT

In [ ]:
# Lancer l'évaluation finale
print("\n Résultats sur le test :")
test_results = trainer_lora.evaluate(tokenized_dataset["test"])
for k, v in test_results.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

In [ ]:
# Récupérer les logs
df_logs = pd.DataFrame(trainer_lora.state.log_history)

# Séparer
df_train = df_logs[df_logs["loss"].notna()].copy()
df_eval  = df_logs[df_logs["eval_loss"].notna()].copy()

# Epochs
df_train["epoch_int"] = np.floor(df_train["epoch"]).astype(int) + 1
df_eval["epoch_int"]  = np.round(df_eval["epoch"]).astype(int)   # eval est à la fin d'epoch

# Agrégation
df_train_epoch = df_train.groupby("epoch_int", as_index=False).agg(train_loss=("loss","mean"))

df_eval_epoch = df_eval.groupby("epoch_int", as_index=False).agg(
    val_loss=("eval_loss","mean"),
    val_accuracy=("eval_accuracy","mean"),
    val_f1_macro=("eval_f1_macro","mean"),
)

# Merge propre (alignement)
df_epoch = pd.merge(df_train_epoch, df_eval_epoch, on="epoch_int", how="outer").sort_values("epoch_int")
display(df_epoch)


# Tracer les courbes
# Loss train vs validation
plt.figure(figsize=(6,4))
plt.plot(df_epoch["epoch_int"], df_epoch["train_loss"],
         marker="o", label="Train loss")
plt.plot(df_epoch["epoch_int"], df_epoch["val_loss"],
         marker="o", label="Validation loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss par epoch (train vs validation)")
plt.grid(True)
plt.legend()
plt.show()


# Accuracy (validation)
plt.figure(figsize=(6,4))
plt.plot(df_epoch["epoch_int"], df_epoch["val_accuracy"],
         marker="o", label="Validation accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy sur validation")
plt.grid(True)
plt.legend()
plt.show()


# F1 macro (validation)
plt.figure(figsize=(6,4))
plt.plot(df_epoch["epoch_int"], df_epoch["val_f1_macro"],
         marker="o", label="Validation F1 macro")

plt.xlabel("Epoch")
plt.ylabel("F1 macro")
plt.title("F1 macro sur validation")
plt.grid(True)
plt.legend()
plt.show()




### **Interprétation**

Les figures montrent une **amélioration nette et régulière** des performances pendant l’entraînement, tant sur l’ensemble d’entraînement que sur la validation.  
- La **train loss** diminue de **0.6337 → 0.2902**, ce qui indique un apprentissage continu et stable du modèle.  
- La **validation loss** diminue globalement de **0.4349 → 0.3152**, avec une **valeur minimale à l’époque 7 (0.3109)** puis une légère remontée jusqu’à l’époque 11.

Cette évolution parallèle jusqu’à l’époque 6–7 suggère que **le modèle généralise bien au début de l’entraînement**. Après l’époque 7 on observe une **légère divergence** : la loss d’entraînement continue de baisser tandis que la loss de validation remonte faiblement, signe d’un début de sur‑ajustement modéré si l’entraînement se prolonge sans régularisation.

Les métriques de validation confirment la bonne dynamique et précisent le compromis :  
- **Accuracy** passe de **0.8297 → 0.8896**, avec un **pic observé autour des époques 9–10** (maximum à l’époque 10 ≈ 0.8905).  
- **F1 macro** passe de **0.8290 → 0.8895**, avec une tendance similaire et un léger pic vers l’époque 10.

**Conclusion** : le meilleur compromis dépend de l’objectif.  
- Si l’on privilégie **minimiser la validation loss** (meilleure généralisation mesurée par la perte), l’**époque 7** est optimale.  
- Si l’on privilégie **les métriques (accuracy / F1)**, les **époques 9–10** offrent les scores les plus élevés au prix d’une loss de validation légèrement supérieure.  

Avec la configuration actuelle, sauvegarde automatique : époque 10 (meilleure F1 macro).

In [ ]:
# Sauvegarde "verrouillée" (best model déjà rechargé)
trainer_lora.save_model(save_path + "/final_model")
tokenizer.save_pretrained(save_path + "/final_model")

# Vérification
print(f"Modèle sauvegardé dans {save_path}/final_model")

print("\n Best metric:", trainer_lora.state.best_metric)
print("\n Best checkpoint:", trainer_lora.state.best_model_checkpoint)

## Matrice de confusion (test)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# On compare prédiction vs vérité terrain
# Prédictions sur le test set
predictions = trainer_lora.predict(tokenized_dataset["test"])

y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=1)

labels = ["entailment", "neutral", "contradiction"]

# Matrice de confusion brute
cm = confusion_matrix(y_true, y_pred)
#cm_df = pd.DataFrame(cm, index=labels, columns=labels)
#cm_df

# Affichage
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
plt.figure(figsize=(6,6))
disp.plot(cmap="Blues", values_format="d")
plt.title("Matrice de confusion – Test (SNLI)")
plt.grid(False)
plt.show()


- Classe 0 — Entailment (réel)

Corrects : 3056

Erreurs :

entailment → neutral : 269

entailment → contradiction : 43

Le modèle identifie très bien les cas d’entailment.
Les erreurs se dirigent majoritairement vers neutral, ce qui est typique : lorsqu’une implication n’est pas parfaitement explicite, le modèle préfère classer en neutral plutôt que de confondre avec contradiction.
Les confusions directes entailment ↔ contradiction restent faibles, ce qui montre une bonne compréhension des relations logiques fortes

- Classe 1 — Neutral (réel)

Corrects : 2775

Erreurs :

neutral → entailment : 249

neutral → contradiction : 195

Neutral est la classe la plus difficile.
Les erreurs sont presque symétriquement réparties entre entailment et contradiction, ce qui reflète la difficulté intrinsèque de la neutralité : elle correspond à une absence de lien logique clair, ce qui peut facilement être interprété comme une implication faible ou une contradiction légère selon le contexte.
Le modèle a donc plus de mal à tracer la frontière entre neutral et les deux autres classes.

- Classe 2 — Contradiction (réel)

Corrects : 2915

Erreurs :

contradiction → neutral : 232

contradiction → entailment : 90

Le modèle reconnaît bien les contradictions explicites.
Les erreurs se dirigent principalement vers neutral, ce qui est logique : certaines contradictions sont subtiles et peuvent ressembler à une simple absence d’information plutôt qu’à une opposition claire.
Les confusions contradiction → entailment restent limitées, ce qui montre que le modèle distingue correctement les relations opposées.

---

La matrice de confusion montre que :

- Le modèle est globalement performant sur les trois classes.

- Les confusions les plus fréquentes concernent la classe neutral, ce qui est parfaitement attendu en NLI.

- Les erreurs directes entre entailment et contradiction sont rares, signe que le modèle comprend bien les relations logiques fortes.

- Les prédictions correctes sont majoritaires dans chaque classe, ce qui confirme la bonne capacité de généralisation observée dans les courbes de loss et les métriques de validation.



# 4) **Evaluation qualitative**

In [ ]:
# NETTOYAGE ET RECHARGEMENT DU MEILLEUR MODÈLE

print("-"*10)
print("RECHARGEMENT DU MEILLEUR MODÈLE")
print("-"*10)

# Libérer la mémoire du modèle d'entraînement
del model_lora
del trainer_lora
torch.cuda.empty_cache()

# Recharger le meilleur checkpoint (modèle + LoRA)
best_checkpoint = "/content/drive/MyDrive/PROJET_DJMA/models/distilbert_lora_snli_nli/checkpoint-171680"

from peft import PeftModel

# Charger le modèle de base
model_base = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

# Charger les adaptateurs LoRA
model = PeftModel.from_pretrained(model_base, best_checkpoint)

# Option A : Garder LoRA (plus rapide pour l'inférence)
model.eval()
model.to(device)

# Option B : Fusionner les poids LoRA dans le modèle
# model = model.merge_and_unload()  # Décommentez si vous voulez fusionner
# model.eval()
# model.to(device)

print(f"Modèle chargé depuis : {best_checkpoint}")
print(f"Device : {model.device}")


In [ ]:
print("\n" + "-"*10)
print("ÉVALUATION QUALITATIVE")
print("-"*10)

# Labels mapping
labels_map = {0: 'entailment', 1: 'neutral', 2: 'contradiction'}
reverse_labels = {'entailment': 0, 'neutral': 1, 'contradiction': 2}

# Exemples de test
test_examples = [
    # Cas évidents d'entailment
    ("A man is eating pizza", "A man is eating food", "entailment"),
    ("The dog is running in the park", "A dog is outside", "entailment"),
    ("A child is playing with a ball", "A kid is playing", "entailment"),

    # Cas évidents de contradiction
    ("A woman is dancing", "A woman is sleeping", "contradiction"),
    ("It's raining heavily", "The sun is shining", "contradiction"),
    ("The cat is black", "The cat is white", "contradiction"),

    # Cas neutres (plus difficiles)
    ("A cat is on a table", "An animal is indoors", "neutral"),
    ("A man is reading a book", "The man is reading a novel", "neutral"),
    ("A woman is cooking", "The woman is making dinner", "neutral"),

    # Cas limites (pour tester les confusions)
    ("A person is running", "Someone is exercising", "entailment"),  # ou neutral ?
    ("The car is red", "The vehicle is red", "entailment"),
    ("It is snowing", "It is cold", "neutral"),  # Inférence probable
    ("A dog is barking", "An animal is making noise", "entailment"),
    ("The building is tall", "The building is 100 meters high", "neutral"),

    # Cas avec négation (difficiles)
    ("Nobody is in the room", "The room is empty", "entailment"),
    ("The movie is not boring", "The movie is exciting", "neutral"),
]

# PRÉDICTIONS ET ANALYSE

correct = 0
total = len(test_examples)
confusion_dict = {label: {label: 0 for label in labels_map.values()} for label in labels_map.values()}

results = []

print("\n RÉSULTATS PAR EXEMPLE:\n")

for premise, hypothesis, expected_label in test_examples:
    # Tokenisation
    inputs = tokenizer(
        premise,
        hypothesis,
        return_tensors="pt",
        truncation=True,
        max_length=128,
        padding=True
    )

    # Move to device
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Prédiction
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)[0]
        predicted_idx = logits.argmax(-1).item()

    predicted_label = labels_map[predicted_idx]
    expected_idx = reverse_labels[expected_label]

    # Statistiques
    is_correct = (predicted_idx == expected_idx)
    if is_correct:
        correct += 1

    confusion_dict[expected_label][predicted_label] += 1

    # Status
    status = "✅" if is_correct else "❌"

    # Affichage
    print(f"{status} [{expected_label.upper():15s} → {predicted_label.upper():15s}]")
    print(f"    Premise   : {premise}")
    print(f"    Hypothesis: {hypothesis}")
    print(f"    Probabilities:")
    print(f"      - Entailment    : {probs[0]:.3f}")
    print(f"      - Neutral       : {probs[1]:.3f}")
    print(f"      - Contradiction : {probs[2]:.3f}")

    if not is_correct:
        print(f"⚠️ Erreur : Attendu {expected_label}, prédit {predicted_label}")

    print()

    # Stocker pour analyse
    results.append({
        'premise': premise,
        'hypothesis': hypothesis,
        'expected': expected_label,
        'predicted': predicted_label,
        'correct': is_correct,
        'prob_entailment': probs[0].item(),
        'prob_neutral': probs[1].item(),
        'prob_contradiction': probs[2].item()
    })



In [ ]:
# Afficher les résultats
print("\n Aperçu des résultats:")
df_qualitative = pd.DataFrame(results)
df_qualitative

In [ ]:
# STATISTIQUES GLOBALES

print("\n" + "-"*10)
print(" STATISTIQUES GLOBALES")
print("-"*10)

accuracy = (correct / total) * 100
print(f"\n✅ Accuracy sur exemples qualitatifs : {accuracy:.1f}% ({correct}/{total})")

# Matrice de confusion simplifiée
print("\n Matrice de confusion (exemples qualitatifs):")
print(f"\n{'':15s} | {'entailment':>12s} | {'neutral':>12s} | {'contradiction':>14s}")
print("-" * 60)
for expected in labels_map.values():
    print(f"{expected:15s} |", end="")
    for predicted in labels_map.values():
        count = confusion_dict[expected][predicted]
        print(f" {count:>12d} |", end="")
    print()

# Analyse par classe
print("\n📈 Performance par classe:")
for label in labels_map.values():
    total_class = sum(confusion_dict[label].values())
    correct_class = confusion_dict[label][label]
    if total_class > 0:
        acc_class = (correct_class / total_class) * 100
        print(f"  - {label:15s}: {acc_class:5.1f}% ({correct_class}/{total_class})")


# ANALYSE DES ERREURS
print("\n" + "-"*10)
print("🔍 ANALYSE DES ERREURS")
print("-"*10)

errors = [r for r in results if not r['correct']]

if errors:
    print(f"\n❌ {len(errors)} erreurs détectées:\n")

    for i, err in enumerate(errors, 1):
        print(f"Erreur {i}:")
        print(f"  Premise   : {err['premise']}")
        print(f"  Hypothesis: {err['hypothesis']}")
        print(f"  Attendu   : {err['expected']}")
        print(f"  Prédit    : {err['predicted']}")
        print(f"  Confiance : {err[f'prob_{err['predicted']}']*100:.1f}%")
        print()
else:
    print("\n🎉 Aucune erreur ! Tous les exemples qualitatifs sont corrects.")


Cette validation reste conditionnée au cadre SNLI (anglais, phrases courtes, annotations humaines). La transposition à DJMA nécessitera une analyse qualitative complémentaire.

---